# From bag-of-words to tf-idf: scaling text features

_Feature Engineering_

**Weight each word by how rare it is across documents, so common words stop drowning out the signal.**

Start with the bag-of-words matrix $X$: one row per document, one column per word, each entry a raw count. tf-idf is nothing more than scaling each column of that matrix by a single number, the word's inverse document frequency (idf).

       The idf is large for rare words (seen in few documents) and small — near zero — for words seen in nearly every document. Multiply a column of counts by its idf and you stretch the rare, distinctive words while flattening the ubiquitous boilerplate.

---

This notebook is a practice scaffold for the **From bag-of-words to tf-idf: scaling text features** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# Lesson: tf-idf — down-weight common words, up-weight rare distinctive ones.
# tf-idf is a COLUMN SCALING of bag-of-words: each word's count column is multiplied by
# idf = log( (#docs) / (#docs containing the word) ) (+1 smoothing). A word in EVERY doc
# gets the smallest idf (here 1.0, its floor) so it barely counts; a word in just 1-2 docs
# gets a big idf (here ~2.0) so it counts a lot.
#
# PROBLEM (raw bag-of-words counts): a filler word like "the" appears MANY times in EVERY
# document. Cosine similarity adds up products of word counts, so those big shared "the"
# counts DOMINATE. Two docs look "most similar" mostly because they both repeat "the" a lot.
# So nearest-neighbor search returns the WRONG, off-topic document: the query (a VOLCANO
# news item) is matched to a filler-heavy OFFICE memo instead of the real volcano doc.
# FIX: TfidfVectorizer. "the" (in every doc) gets its floor weight, so it stops driving
# similarity; the rare distinctive shared words ("volcano", "lava", "crater", "eruption")
# now dominate, and nearest-neighbor returns the CORRECT topical volcano match.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---- STEP 1: Load real data (a small REAL inline corpus) ---------------------
# 7 short English docs. Doc 0 is the QUERY: a volcano news item, padded (as real prose is)
# with lots of "the ... and ...". Doc 1 is its TRUE topical twin (also a volcano item: it
# shares the rare words volcano/lava/ash/crater/eruption, but uses little filler). Docs 2-6
# are OFF-TOPIC memos (office, finance, sports, cars, pets), each padded with the SAME heavy
# "the"/"and" filler as the query. Deterministic: a fixed inline corpus, no randomness.
docs = [
    "the lava and the ash and the crater and the smoke and the volcano and the eruption today",  # 0 QUERY: volcano
    "volcano lava ash crater eruption and the smoke",                                             # 1 TRUE match: volcano twin
    "the memo and the report and the budget and the meeting and the office and the plan today",   # 2 office (filler-heavy)
    "the bank and the loan and the rate and the cash and the market and the stock today",         # 3 finance
    "the team and the game and the score and the player and the goal and the coach today",        # 4 sports
    "the car and the road and the wheel and the engine and the speed and the brake today",        # 5 cars
    "the dog and the park and the walk and the leash and the treat and the owner today",          # 6 pets
]
labels = ["0 QUERY (volcano)", "1 volcano", "2 office", "3 finance", "4 sports", "5 cars", "6 pets"]
short = [l.split()[0] + " " + l.split()[1].strip("()") for l in labels]  # e.g. "0 QUERY"

# ---- STEP 2: Visualize the PURE (raw) data ----------------------------------
# Build the raw bag-of-words count matrix and look at a COMMON word ("the") vs a RARE
# distinctive word ("volcano") in every doc. "the" appears 6x in nearly every doc and
# towers over everything; "volcano" is a tiny spike present in only docs 0 and 1.
count_vec = CountVectorizer()
X_counts = count_vec.fit_transform(docs)
vocab = count_vec.vocabulary_
the_counts = X_counts[:, vocab["the"]].toarray().ravel()
volcano_counts = X_counts[:, vocab["volcano"]].toarray().ravel()

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
xpos = np.arange(len(docs))
ax[0].bar(xpos - 0.2, the_counts, width=0.4, color="#7f8c8d", label="'the' (common)")
ax[0].bar(xpos + 0.2, volcano_counts, width=0.4, color="#c0392b", label="'volcano' (rare)")
ax[0].set_title("STEP 2: raw counts per doc\n'the' DROWNS OUT 'volcano'")
ax[0].set_xticks(xpos); ax[0].set_xticklabels(short, rotation=30, ha="right")
ax[0].set_xlabel("document"); ax[0].set_ylabel("word count"); ax[0].legend()

# ---- STEP 3: Reproduce the PROBLEM on the raw counts -------------------------
# Cosine similarity of the QUERY (doc 0) against every other doc, using RAW counts.
# Every filler-heavy memo shares the query's big "the" mass, so an OFF-TOPIC memo looks
# most similar and the real volcano twin (doc 1, low on filler) ranks LAST. WRONG.
raw_sims = cosine_similarity(X_counts[0], X_counts).ravel()
raw_order = np.argsort(-raw_sims)
raw_nn = raw_order[raw_order != 0][0]  # nearest doc that is not the query itself
print("STEP 3 PROBLEM - nearest-neighbor by RAW-count cosine similarity:")
for j in raw_order:
    tag = "  <-- QUERY" if j == 0 else ("  <-- picked (WRONG)" if j == raw_nn else "")
    print(f"   sim={raw_sims[j]:.3f}  doc {labels[j]}{tag}")
print(f"   raw picks doc {raw_nn} ({labels[raw_nn]}) - off-topic filler, NOT the volcano twin (doc 1).\n")

# ---- STEP 4: Apply tf-idf, then visualize the engineered data ---------------
# tf-idf re-scales each word column by idf. Compare the WEIGHT a common vs rare word carries.
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(docs)
idf = tfidf_vec.idf_
tvocab = tfidf_vec.vocabulary_
idf_the, idf_volcano = idf[tvocab["the"]], idf[tvocab["volcano"]]

# Bar: a common word's vs a rare word's "weight" under raw counts vs tf-idf.
# "Raw weight" = how much that word can contribute (its max count in any doc); under raw
# counts "the" carries a huge weight. Under tf-idf, "the" drops to its idf floor while
# "volcano" gets boosted by its high idf.
raw_w_the, raw_w_volcano = the_counts.max(), volcano_counts.max()
ax[1].bar([0, 1], [raw_w_the, raw_w_volcano], width=0.4, color="#7f8c8d", label="raw counts")
ax[1].bar([0.4, 1.4], [idf_the, idf_volcano], width=0.4, color="#27ae60", label="tf-idf (idf)")
ax[1].set_xticks([0.2, 1.2]); ax[1].set_xticklabels(["'the'\n(common)", "'volcano'\n(rare)"])
ax[1].set_title("STEP 4: word weight, raw vs tf-idf\n'the' falls to its floor; 'volcano' boosted")
ax[1].set_ylabel("weight"); ax[1].legend()
print(f"STEP 4 - idf('the')={idf_the:.2f} (its floor; appears in every doc)   "
      f"idf('volcano')={idf_volcano:.2f} (high; appears in only 2 docs)")

# ---- STEP 5: Show the FIX ---------------------------------------------------
# Same cosine similarity, same query, now on tf-idf vectors. The shared "the" mass no
# longer dominates, so the rare topical words decide it: the true volcano twin (doc 1)
# is correctly picked as nearest neighbor.
eng_sims = cosine_similarity(X_tfidf[0], X_tfidf).ravel()
eng_order = np.argsort(-eng_sims)
eng_nn = eng_order[eng_order != 0][0]
print("\nSTEP 5 FIX - nearest-neighbor by tf-idf cosine similarity:")
for j in eng_order:
    tag = "  <-- QUERY" if j == 0 else ("  <-- picked (CORRECT)" if j == eng_nn else "")
    print(f"   sim={eng_sims[j]:.3f}  doc {labels[j]}{tag}")

# Plot: similarity to the query, raw vs tf-idf, side by side per non-query doc.
others = [j for j in range(len(docs)) if j != 0]
ax[2].bar(np.array(others) - 0.2, raw_sims[others], width=0.4, color="#7f8c8d", label="raw counts")
ax[2].bar(np.array(others) + 0.2, eng_sims[others], width=0.4, color="#27ae60", label="tf-idf")
ax[2].set_title(f"STEP 5: cosine sim to QUERY (doc 0)\nraw picks {raw_nn}, tf-idf picks {eng_nn} (volcano)")
ax[2].set_xticks(others); ax[2].set_xticklabels([short[j] for j in others], rotation=30, ha="right")
ax[2].set_xlabel("document"); ax[2].set_ylabel("cosine similarity to query"); ax[2].legend()
plt.tight_layout(); plt.show()

print(f"\nPROBLEM (raw): nearest = doc {raw_nn} ({labels[raw_nn]})   ->   "
      f"FIX (tf-idf): nearest = doc {eng_nn} ({labels[eng_nn]})")

## Reference implementation — scikit-learn

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfTransformer, TfidfVectorizer)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

# Yelp reviews (Dataset Challenge round 6). Get the data from
# yelp.com/dataset; prepared notebooks: github.com/alicezheng/feature-engineering-book
# A binary task: predict whether a review is for a restaurant (1) or not (0).
review_df = pd.read_json('yelp_academic_dataset_review.json', lines=True)
texts  = review_df['text']
labels = review_df['target']        # 0/1 class column prepared as in the book

train_x, test_x, train_y, test_y = train_test_split(
    texts, labels, train_size=0.7, random_state=123)

# 1) BAG-OF-WORDS: raw counts. Fit the vocabulary on TRAIN only.
bow = CountVectorizer()
X_tr_bow = bow.fit_transform(train_x)
X_te_bow = bow.transform(test_x)            # reuse train vocabulary

# 2) L2-NORMALIZED bag-of-words: scale each document row to unit length.
l2 = TfidfTransformer(norm='l2', use_idf=False)   # norm only, no idf
X_tr_l2 = l2.fit_transform(X_tr_bow)
X_te_l2 = l2.transform(X_te_bow)

# 3) TF-IDF: column-scale the counts by inverse document frequency.
#    idf is FIT ON TRAIN; the test set reuses the train idf.
tfidf = TfidfTransformer()                  # norm='l2', use_idf=True by default
X_tr_tfidf = tfidf.fit_transform(X_tr_bow)
X_te_tfidf = tfidf.transform(X_te_bow)
# (TfidfVectorizer() == CountVectorizer() + TfidfTransformer() in one step.)

def tuned_logreg(Xtr, ytr, Xte, yte):
    """GridSearchCV over the regularization strength C, then test accuracy."""
    grid = {'C': [1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]}
    search = GridSearchCV(LogisticRegression(max_iter=1000), grid, cv=5)
    search.fit(Xtr, ytr)
    return search.best_params_['C'], search.score(Xte, yte)

for name, Xtr, Xte in [("bag-of-words",       X_tr_bow,   X_te_bow),
                       ("L2-normalized BoW",  X_tr_l2,    X_te_l2),
                       ("tf-idf",             X_tr_tfidf, X_te_tfidf)]:
    best_C, acc = tuned_logreg(Xtr, train_y, Xte, test_y)
    print(f"{name:<20} best C={best_C:<6}  test accuracy={acc:.4f}")

# The book's takeaway: the three accuracies land CLOSE together, and the
# choice of C (regularization) moves the score more than the scaling does.

## Visualize the data & results

_On a tiny real corpus, what happens to a COMMON word's weight versus a RARE word's weight when we switch from raw counts to tf-idf?_

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# A small REAL inline corpus: 'the' is common across all docs;
# 'volcano' is rare and specific to one doc.
corpus = [
    "the food the the the was good",          # 'the' x4
    "the service was the slow",
    "the place was the busy the",
    "the menu was the long",
    "the volcano roll was the special",       # 'volcano' x1, only here
]

bow = CountVectorizer()
X = bow.fit_transform(corpus).toarray()       # raw bag-of-words counts
vocab = bow.vocabulary_
N = X.shape[0]                                 # number of documents

# The book's plain idf = log(N / n_w), natural log, no smoothing.
# tf-idf is exactly a COLUMN-SCALING of the count matrix by idf.
df  = (X > 0).sum(axis=0)                      # document frequency per word
idf = np.log(N / df)                          # idf(w) = log(N / n_w)
Xt  = X * idf                                  # column-scale -> tf-idf

def weights(word):
    j = vocab[word]
    return X[:, j].max(), Xt[:, j].max()      # raw count, tf-idf weight

the_raw,  the_tfidf  = weights("the")         # 4.0 , 0.0   (idf = log(5/5)=0)
volc_raw, volc_tfidf = weights("volcano")     # 1.0 , 1.609 (idf = log(5/1))
print([round(the_raw,2), round(the_tfidf,2),
       round(volc_raw,2), round(volc_tfidf,2)])
# -> [4.0, 0.0, 1.0, 1.61]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A corpus has $N=1000$ documents. Word "service" appears in 900 of them; word "mediocre" appears in 10. Using the book's plain idf $\log(N/n_w)$ (natural log), which word does tf-idf trust more, and by how much per occurrence?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute idf("service") = log(1000/900). — _Document frequency 900 out of 1000 means the word is nearly ubiquitous, so the ratio is barely above 1._
- Compute idf("mediocre") = log(1000/10). — _Document frequency 10 out of 1000 means the word is rare, so the ratio is 100 and its log is large._
- Compare the two idf values. — _Each occurrence of the word contributes (count × idf), so the idf is the per-occurrence weight._

**Answer:** $\text{idf}(\text{service})=\log(1000/900)=\log(1.111)\approx 0.105$. $\text{idf}(\text{mediocre})=\log(1000/10)=\log(100)\approx 4.605$. tf-idf trusts "mediocre" about $4.605/0.105 \approx 44\times$ more per occurrence — the rare, distinctive word dominates even though "service" is counted far more often across the corpus.

</details>

**Problem 2.** You vectorize your training reviews with tf-idf and get 0.93 cross-validated accuracy. Eager to report test accuracy, you call TfidfVectorizer().fit_transform(test_reviews) and the score jumps. Why is this wrong, and what should you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice that fit_transform re-learns the idf on the test set. — _The idf weights (and the vocabulary) are now computed from test documents, which the model is supposed to have never seen._
- Recognize this as data leakage. — _Test-set statistics have bled into the features; the inflated score does not reflect real-world performance on unseen text._
- Use the train-fitted vectorizer to only transform the test set. — _The book's rule: idf is fit on train only; test documents reuse the train idf and vocabulary._

**Answer:** It is leakage: fit_transform on the test set recomputes the idf (and vocabulary) from data the model must not see, so the score is optimistic and meaningless. Fit the vectorizer once on the training reviews — vec.fit(train) — then call vec.transform(test) so the test documents reuse the train idf. In a pipeline, put the vectorizer inside the cross-validation so it is refit on each training fold only.

</details>